# Модуль 0. Зачем валидировать модели

**Длительность:** 60 минут  
**Формат:** теория (45 мин) + практика (15 мин)  
**Цель модуля:** Учащийся должен понять, что хороший результат на обучающих данных не является доказательством работоспособности модели в реальности, и осознать необходимость отдельной оценки на данных, которые модель не видела.

## 1. Введение: постановка задачи (5 мин)

### Что объяснить
Машинное обучение в контексте предсказания решает следующую задачу: имеется набор объектов с известными признаками и известным целевым значением. Требуется построить функцию, которая по признакам нового, ранее не виденного объекта выдает предсказание целевого значения.

Ключевое требование: функция должна работать не только на тех объектах, которые использовались при построении, но и на новых объектах из того же источника.

### Что подчеркнуть
Если модель работает только на уже известных ей объектах, она не выполняет свою задачу. Она просто запомнила ответы. Нам нужна модель, которая выявила закономерность.

## 2. Обучающая выборка и обобщающая способность (15 мин)

### 2.1. Определение обучающей выборки
**Обучающая выборка (training set)** — это набор пар «признаки -> целевое значение», на котором модель настраивает свои внутренние параметры.

Процесс обучения заключается в том, чтобы алгоритм подобрал такие параметры, при которых его предсказания на этой выборке максимально близки к истинным значениям.

### 2.2. Почему идеальная точность на обучении не гарантирует успех

Точность на обучающей выборке измеряет, насколько хорошо модель воспроизводит уже известные ей данные. Это не эквивалентно способности предсказывать на новых данных по двум причинам:

1. **Случайный шум.** В любых реальных данных присутствует случайная составляющая, не несущая предсказательной ценности. Если модель слишком гибкая, она начинает подстраиваться под этот шум, принимая его за закономерность.
2. **Специфика выборки.** Конкретный набор данных — это лишь конечное подмножество всех возможных объектов. Оно может содержать локальные особенности, которые не повторятся в другой выборке из того же процесса.

### 2.3. Обобщающая способность
**Обобщающая способность (generalization)** — это способность модели выдавать корректные предсказания на объектах, которые не участвовали в обучении.

Формально: пусть данные порождены некоторым вероятностным распределением $P(X, Y)$. Обучающая выборка $D_{train}$ — это $n$ реализаций из этого распределения. Хорошая модель минимизирует ожидаемую ошибку на всем распределении, а не только на $D_{train}$.

$$\text{Идеальная цель: } \min_f \mathbb{E}_{(x,y) \sim P} \left[ L(f(x), y) \right]$$

На практике мы не знаем полного распределения $P$, поэтому минимизируем эмпирическую ошибку на обучающей выборке:

$$\min_f \frac{1}{n} \sum_{i=1}^{n} L(f(x_i), y_i)$$

Проблема: минимизация эмпирической ошибки не всегда ведет к минимизации ожидаемой ошибки на всем распределении.

### 2.4. Запоминание против обобщения
- **Запоминание:** модель хранит в параметрах информацию о конкретных объектах обучающей выборки. Пример: таблица соответствия «если объект №47, то ответ 1».
- **Обобщение:** модель выявила функциональную зависимость между признаками и целевой переменной, которая сохраняется за пределами обучающей выборки.

Мы строим модели для обобщения. Валидация — это процедура проверки, достигнуто ли обобщение.

## 3. Переобучение (Overfitting) (15 мин)

### 3.1. Определение
**Переобучение** — это ситуация, в которой модель слишком сильно адаптирована к обучающей выборке, включая её случайные шумы и локальные особенности, в результате чего её качество на новых данных существенно ниже, чем на обучающих.

Формальный признак:
$$\text{Error}_{train} \ll \text{Error}_{new}$$

где $\text{Error}_{train}$ — ошибка на обучении, $\text{Error}_{new}$ — ошибка на новых данных.

### 3.2. Причина переобучения
Переобучение возникает при избыточной сложности модели относительно объема и информативности данных.

**Сложность модели** определяется её емкостью (capacity) — количеством различных функций, которые модель способна реализовать. Например:
- Линейная регрессия: низкая емкость (только линейные зависимости).
- Дерево решений без ограничения глубины: высокая емкость (может построить отдельное правило для каждого объекта).

Если емкость модели позволяет достичь нулевой ошибки на обучении, но данных недостаточно для устойчивого определения границы между сигналом и шумом, модель подстроится под шум.

### 3.3. Признаки переобучения
1. Значительный разрыв между метрикой на обучении и метрикой на отложенных данных.
2. Рост метрики на обучении при увеличении сложности модели при одновременном падении метрики на новых данных.
3. Модель выдает крайне неустойчивые предсказания: небольшое изменение входных данных приводит к резкому изменению ответа.

### 3.4. Почему это критично
В индустрии модель обучается один раз, а применяется многократно к новым объектам. Переобученная модель демонстрирует высокие метрики на этапе разработки и проваливается в эксплуатации. Это приводит к финансовым потерям, некорректным решениям и потере доверия к системе.

## 4. Параметры модели и гиперпараметры (10 мин)

### 4.1. Параметры модели
**Параметры модели** — это внутренние переменные, которые алгоритм оптимизирует в процессе обучения на обучающей выборке.

Примеры:
- Веса в линейной регрессии: $w_0, w_1, \dots, w_m$.
- Структура дерева решений: пороги разбиений в узлах, признаки для разбиения, метки в листьях.
- Веса нейронной сети.

Параметры изменяются автоматически в ходе оптимизации целевой функции (например, минимизации среднеквадратичной ошибки).

### 4.2. Гиперпараметры
**Гиперпараметры** — это переменные, которые задаются до начала обучения и определяют процесс оптимизации или структуру модели.

Примеры:
- `max_depth` в DecisionTreeClassifier — максимальная глубина дерева.
- `n_neighbors` в KNN — количество соседей.
- `learning_rate` в градиентном бустинге — шаг градиентного спуска.

Гиперпараметры не обучаются на данных. Их значения подбираются отдельно.

### 4.3. Связь с переобучением
Гиперпараметры напрямую контролируют емкость модели и, следовательно, риск переобучения.

Пример: увеличение `max_depth` в дереве решений повышает емкость модели. При `max_depth=1` модель может реализовать только простое правило. При неограниченной глубине модель может построить дерево, где каждый лист содержит ровно один объект обучающей выборки — полное запоминание.

Выбор гиперпараметров требует оценки качества на данных, не участвовавших в обучении. Именно поэтому нам нужна валидация.

## 5. Практическая часть: демонстрация переобучения (15 мин)

### 5.1. Постановка эксперимента
Сгенерируем синтетический датасет с двумя признаками и бинарной целевой переменной. Данные будут иметь некоторую базовую зависимость, но с добавлением случайного шума.

Обучим DecisionTreeClassifier с различными значениями `max_depth`: 1, 3, 5, 10, 20 и `None` (без ограничений). Для каждой модели измерим accuracy на обучающей выборке и на отложенной тестовой выборке.

### 5.2. Ожидаемый результат
- При малых глубинах (1, 3) accuracy на обучении и тесте будет невысоким и близким друг к другу. Модель недостаточно сложна — **недообучение** (underfitting).
- При умеренных глубинах (5, 10) accuracy на обучении растет, accuracy на тесте тоже растет или стабилизируется на максимуме. Модель уловила закономерность.
- При больших глубинах (20, None) accuracy на обучении достигает 100 % или близко к нему, а accuracy на тесте падает. Это и есть переобучение.

### 5.3. Код для практики

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# 1. Генерация данных
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

# Разбиение на обучение и тест
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 2. Перебор глубин дерева
depths = [1, 3, 5, 10, 20, None]
train_scores = []
test_scores = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    train_scores.append(train_acc)
    test_scores.append(test_acc)
    
    print(f"max_depth={depth}: Train={train_acc:.3f}, Test={test_acc:.3f}")

# 3. Визуализация
depth_labels = [str(d) if d is not None else "None" for d in depths]

plt.figure(figsize=(8, 5))
plt.plot(depth_labels, train_scores, marker='o', label='Train Accuracy')
plt.plot(depth_labels, test_scores, marker='s', label='Test Accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Переобучение: точность на обучении и тесте')
plt.legend()
plt.ylim(0.5, 1.05)
plt.grid(True)
plt.show()

### 5.4. Что обсудить по результатам
- При `max_depth=None` дерево достигает 100 % на обучении. Это означает, что модель построила правила, уникальные для каждого объекта обучающей выборки.
- Точность на тесте при этом ниже. Модель перенесла на тест шум и специфику обучающей выборки, которые там не повторяются.
- Оптимальная глубина — та, при которой тестовая точность максимальна. Это и есть компромисс между сложностью и обобщением.

## 6. Итоги модуля (5 мин)

### Ключевые тезисы
1. Модель обучается на конечной выборке, но должна работать на всем распределении данных.
2. Высокое качество на обучающей выборке не доказывает, что модель выявила закономерность. Она могла запомнить данные.
3. Переобучение — это адаптация к шуму и локальным особенностям обучающей выборки. Его признак: большой разрыв между метриками на обучении и новых данных.
4. Гиперпараметры управляют сложностью модели и риском переобучения. Их выбор требует оценки на данных, которые модель не видела.
5. Валидация — это процедура оценки качества модели на отдельной выборке для проверки обобщающей способности.

### Контрольные вопросы
- Почему модель с accuracy 100 % на обучении может оказаться бесполезной?
- Что такое обобщающая способность и чем она отличается от запоминания?
- Как гиперпараметр `max_depth` влияет на риск переобучения?
- В чем формальный признак переобучения?

### Что дальше
В следующем модуле мы изучим, как правильно разбивать данные для получения объективной оценки качества: от простого hold-out до кросс-валидации и стратифицированного разбиения.